In [3]:
import os
import time
from datetime import datetime, timedelta, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import dotenv
import pandas as pd
import requests

dotenv.load_dotenv()

LANGFUSE_PUBLIC_KEY = os.environ["LANGFUSE_PUBLIC_KEY"]
LANGFUSE_SECRET_KEY = os.environ["LANGFUSE_SECRET_KEY"]
LANGFUSE_HOST = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")

TRACES_URL = f"{LANGFUSE_HOST}/api/public/traces"
SCORES_URL = f"{LANGFUSE_HOST}/api/public/v2/scores"

AUTH = (LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY)
session = requests.Session()


def iso_z(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


def day_ranges(days_back: int, step_days: int = 1) -> list[tuple[str, str]]:
    today = datetime.now(timezone.utc).date()
    start = today - timedelta(days=days_back)

    ranges = []
    cur = start
    while cur <= today:
        end = min(cur + timedelta(days=step_days - 1), today)
        ranges.append(
            (
                iso_z(datetime.combine(cur, datetime.min.time(), tzinfo=timezone.utc)),
                iso_z(datetime.combine(end, datetime.max.time(), tzinfo=timezone.utc)),
            )
        )
        cur = end + timedelta(days=1)
    return ranges


def fetch_paginated(
    url: str,
    params: dict,
    *,
    limit: int = 100,
    max_items: int = 5000,
    max_retries: int = 5,
    timeout: int = 30,
) -> list[dict]:
    rows = []
    page = 1
    limit = min(max(1, limit), 100)

    while len(rows) < max_items:
        attempt = 0

        while True:
            try:
                r = session.get(
                    url,
                    auth=AUTH,
                    params={**params, "limit": limit, "page": page},
                    timeout=timeout,
                )

                if r.status_code == 429:
                    sleep_s = float(r.headers.get("Retry-After", "1"))
                    time.sleep(sleep_s)
                    continue

                if 500 <= r.status_code < 600:
                    if attempt >= max_retries:
                        r.raise_for_status()
                    time.sleep(min(2 ** attempt, 30))
                    attempt += 1
                    continue

                r.raise_for_status()
                payload = r.json()
                break

            except (requests.Timeout, requests.ConnectionError) as e:
                if attempt >= max_retries:
                    raise
                time.sleep(min(2 ** attempt, 30))
                attempt += 1

        chunk = payload.get("data") or []
        if not chunk:
            break

        rows.extend(chunk)

        meta = payload.get("meta") or {}
        current_page = meta.get("page", page)
        total_pages = meta.get("totalPages", page)

        if current_page >= total_pages:
            break

        page += 1

    return rows[:max_items]


def fetch_traces_slice(from_ts: str, to_ts: str, limit: int = 100) -> list[dict]:
    raw = fetch_paginated(
        TRACES_URL,
        {
            "tags": "batch_evaluation",
            "fromTimestamp": from_ts,
            "toTimestamp": to_ts,
        },
        limit=limit,
    )

    return [
        {
            "trace_id": t.get("id"),
            "trace_timestamp": t.get("timestamp") or t.get("createdAt"),
            "model": (t.get("metadata") or {}).get("model", "unknown"),
            "custom_id": (t.get("metadata") or {}).get("custom_id"),
            "batch_eval": bool((t.get("metadata") or {}).get("batch_eval")),
            "run_id": (t.get("metadata") or {}).get("run_id"),
        }
        for t in raw
    ]


def fetch_scores_slice(from_ts: str, to_ts: str, limit: int = 100) -> list[dict]:
    raw = fetch_paginated(
        SCORES_URL,
        {
            "name": "accuracy",
            "tags": "batch_evaluation",
            "fromTimestamp": from_ts,
            "toTimestamp": to_ts,
        },
        limit=limit,
    )

    return [
        {
            "score_id": s.get("id"),
            "trace_id": s.get("traceId"),
            "timestamp": s.get("timestamp") or s.get("createdAt"),
            "accuracy": s.get("value"),
        }
        for s in raw
    ]


def fetch_batch_traces(
    from_days: int = 7,
    slice_days: int = 1,
    per_request_limit: int = 100,
    max_workers: int = 2,
) -> pd.DataFrame:
    ranges = day_ranges(from_days, slice_days)

    trace_rows = []
    score_rows = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {}

        for start, end in ranges:
            futures[ex.submit(fetch_traces_slice, start, end, per_request_limit)] = "traces"
            futures[ex.submit(fetch_scores_slice, start, end, per_request_limit)] = "scores"

        for fut in as_completed(futures):
            kind = futures[fut]
            rows = fut.result()
            if kind == "traces":
                trace_rows.extend(rows)
            else:
                score_rows.extend(rows)

    traces_df = (
        pd.DataFrame(trace_rows).drop_duplicates(subset=["trace_id"])
        if trace_rows
        else pd.DataFrame(columns=["trace_id", "trace_timestamp", "model", "custom_id", "batch_eval", "run_id"])
    )

    scores_df = (
        pd.DataFrame(score_rows)
        if score_rows
        else pd.DataFrame(columns=["score_id", "trace_id", "timestamp", "accuracy"])
    )

    df = scores_df.merge(traces_df, on="trace_id", how="left")
    df["model"] = df["model"].fillna("unknown").astype(str)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    df["date"] = df["timestamp"].dt.date

    return df


df_traces = fetch_batch_traces(
    from_days=37,
    slice_days=1,
    per_request_limit=100,
    max_workers=10,
)

In [4]:
df_traces

,score_id,trace_id,timestamp,accuracy,trace_timestamp,model,custom_id,batch_eval,run_id,date
0,00b34e0f431d4046,c759afeb785c9006deb514c6cbbf8bda,2026-03-03 19:45:04.091000+00:00,0,2026-03-03T19:45:04.090Z,claude-3-5-haiku-20241022,$hash_QmvRf9JKjB+SKkky7OkZFA==,True,local-20260303-162342Z,2026-03-03
1,b1ba7f77b6b9c543,9ad43c0c2bc817c843ae97646089068f,2026-03-03 19:45:04.090000+00:00,0,2026-03-03T19:45:04.089Z,claude-3-5-haiku-20241022,7554699334005050646,True,local-20260303-162342Z,2026-03-03
2,300535d2b6e418e0,0b68acf2d1f364ca1a3c03563c613bb4,2026-03-03 19:45:04.090000+00:00,0,2026-03-03T19:45:04.090Z,claude-3-5-haiku-20241022,$hash_wjWYhyzLVvIL8G3L7b6lug==,True,local-20260303-162342Z,2026-03-03
3,6c89c989a7464416,ad0c1162a025811366264bfc07fcd672,2026-03-03 19:45:04.089000+00:00,0,2026-03-03T19:45:04.089Z,claude-3-5-haiku-20241022,3730201357861945585,True,local-20260303-162342Z,2026-03-03
4,40fab153e580bede,8e363c4b52fc191aa7ae09d4d5fdd386,2026-03-03 19:45:04.083000+00:00,0,2026-03-03T19:45:04.082Z,claude-3-5-haiku-20241022,3730196425838441078,True,local-20260303-162342Z,2026-03-03
...,...,...,...,...,...,...,...,...,...,...
38858,ef109c02-eb7e-46ee-bb82-1000fc241fde,5e6dddf6-b9e7-4a7c-9a44-532e1bd0745b,2026-04-01 04:54:30.977000+00:00,1,2026-04-01T04:54:30.977Z,gpt-5.4,3733926866022333336,True,gha-23832575964,2026-04-01
38859,badeb9c2-4e3f-4b00-bebb-d70ec2c39da6,79095c7c-d2d2-4d16-b2f7-0ad0ab1f074b,2026-04-01 04:54:30.977000+00:00,1,2026-04-01T04:54:30.976Z,gpt-5.4,3737578749609921904,True,gha-23832575964,2026-04-01
38860,71fdc4b1-a0ad-4b4c-9197-6c496fb9123d,9769f268-ae87-4ef5-86c9-1103d5b822d5,2026-04-01 04:54:30.976000+00:00,1,2026-04-01T04:54:30.975Z,gpt-5.4,3731716582586975695,True,gha-23832575964,2026-04-01
38861,b7b79bf8-c0f5-4217-a2a4-0908e0de821b,2a213e56-ba50-40f5-b3ec-4a9f4516a07f,2026-04-01 04:54:30.975000+00:00,1,2026-04-01T04:54:30.970Z,gpt-5.4,3730374792961330564,True,gha-23832575964,2026-04-01


In [6]:
# gpt 5 mini model
df_5mini = df_traces[df_traces["model"].str.contains("claude-sonnet-4-6", case=False, na=False)]
# macnemar test of accuracy for gpt-5-mini trace timstamp 08.03 and 10.03

from datetime import date
import math

# select only the two dates of interest
_date_a = date(2026, 3, 8)
_date_b = date(2026, 4, 1)

_df_5mini_dates = df_5mini[df_5mini["timestamp"].dt.date.isin([_date_a, _date_b])].copy()
_df_5mini_dates["date"] = _df_5mini_dates["timestamp"].dt.date

# pivot to have one row per custom_id with accuracies for both days
_pivot = (
    _df_5mini_dates
    .pivot_table(index="custom_id", columns="date", values="accuracy")
    .dropna(subset=[_date_a, _date_b])
    .astype(int)
)

_y_a = _pivot[_date_a]
_y_b = _pivot[_date_b]

# contingency table counts
_a = ((_y_a == 1) & (_y_b == 1)).sum()
_b = ((_y_a == 1) & (_y_b == 0)).sum()
_c = ((_y_a == 0) & (_y_b == 1)).sum()
_d = ((_y_a == 0) & (_y_b == 0)).sum()

_n = _b + _c

# exact binomial version of McNemar's test (no external deps)
def _mcnemar_exact_pvalue(b: int, c: int) -> float:
    n = b + c
    if n == 0:
        return float("nan")
    k = min(b, c)
    p = 0.0
    for i in range(0, k + 1):
        p += math.comb(n, i)
    p = 2 * p / (2 ** n)
    return min(p, 1.0)

_mcnemar_p = _mcnemar_exact_pvalue(_b, _c)
_chi2 = ((abs(_b - _c) - 1) ** 2 / _n) if _n > 0 else float("nan")

print("McNemar contingency table (a, b; c, d):", [[_a, _b], [_c, _d]])
print("discordant pairs b, c:", _b, _c)
print("McNemar chi^2 (with continuity correction):", _chi2)
print("McNemar exact binomial p-value:", _mcnemar_p)


McNemar contingency table (a, b; c, d): [[np.int64(254), np.int64(2)], [np.int64(2), np.int64(9)]]
discordant pairs b, c: 2 2
McNemar chi^2 (with continuity correction): 0.25
McNemar exact binomial p-value: 1.0
